# HANDS-ON MACHINE LEARNING

Vamos fazer uma atividade prática utilizando dados de expressão do TCGA para 3 tecidos:

- COAD (Colon Adenocarcinoma)
- READ (Rectum Adenocarcinoma)
- STAD (Stomach Adenocarcinoma)

Nosso objetivo é verificar se nós conseguimos classificar corretamente 300 amostras aleatórias do TCGA nos tecidos acima mencionados utilizando um modelo específico de machine learning supervisionado, a regressão logística (sim, regressão logística também é machine learning).

## Parte 1 - Processamento dos dados

### Importação de bibliotecas

Primeiro, precisamos importar algumas bibliotecas

In [2]:
# import libraries for processing
import os
import numpy as np
import pandas as pd
from statsmodels import robust

/usr/local/lib/python3.7/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


### Pré-processamento dos dados

Agora, precisamos abrir os arquivos ".tsv" e selecionar, aleatoriamente, 100 amostras de cada tecido. Nesse exemplo, usamos 100 amostras para termos um tamanho amostral robusto para as análises e para fins de tempo de processamento.

In [3]:
# open files and get a 100 sample from each
# get current directory
dirpath = os.getcwd()

# read files and get 100 random samples from each
df_COAD = pd.read_csv(os.path.join(dirpath + "/data/GastroTCGA/COAD_PANCAN.tsv"), delimiter='\t', index_col=0, header=None)
df_COAD = df_COAD.sample(100, axis = 1)
df_READ = pd.read_csv(os.path.join(dirpath + "/data/GastroTCGA/READ_PANCAN.tsv"), delimiter='\t', index_col=0, header=None)
df_READ = df_READ.sample(100, axis = 1)
df_STAD = pd.read_csv(os.path.join(dirpath + "/data/GastroTCGA/STAD_PANCAN.tsv"), delimiter='\t', index_col=0, header=None)
df_STAD = df_STAD.sample(100, axis = 1)

# merge files
df = pd.concat([df_COAD, df_READ, df_STAD], axis=1, ignore_index = True)

FileNotFoundError: ignored

### Seleção de genes

Nessa etapa, nós primeiro calculamos o desvio absoluto da mediana (MAD) da expressão de todos os milhares de genes analisados para os 300 pacientes previamente selecionados.

Depois, selecionamos apenas aqueles genes cujo MAD seja alto o suficiente para potencialmente poder explicar a diferença tecidual (COAD, READ ou STAD) entre as amostras.

In [ ]:
# calculate mad for all genes
gene_mad = robust.mad(df.iloc[1:].values.astype(np.float), axis = 1)

# prepare final dataframe
new_df = pd.DataFrame()
new_df = new_df.append(df.iloc[0], ignore_index = True)

# get only genes with mad equal or greater then 0.94
for i in range(len(gene_mad)):
	if gene_mad[i] >= 0.94:
		new_df=new_df.append(df.iloc[i+1])

# save to file
new_df.to_csv(os.path.join(dirpath + "/data/GastroTCGA.tsv"), sep = '\t')

## Parte 2 - Classificador

Nessa parte é onde a regressão logística de fato ocorre para classificarmos nossas 300 amostras aleatórias em COAD, READ ou STAD de acordo com a expressão gênica.

### Importação de bibliotecas

Novamente precisamos importar mais algumas outras bibliotecas.

A partir daqui começaremos a usar algumas das muitas funcionalidades do SKLEARN. Se tiver curiosidade, acesse o __[link](https://scikit-learn.org/stable/)__ para explorá-lo melhor!

In [ ]:
# import libraries for plotting
import seaborn as sns
import matplotlib.pyplot as plt

# import SKLEARN libraries for machine learning
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict

# import internal library for plotting (check our material for more details)
from plot_confusion import plot_confusion_matrix

### Preparação dos dados para a classificação

Nessa breve etapa, primeiro nós abrimos o arquivo mesclado dos nossos 300 casos aleatórios de pacientes com COAD, READ e STAD com os dados de expressão de genes com MAD elevado.

Depois, precisamos extrair as "características" a serem utilizadas no modelo (expressão gênica - cada gene é uma característica) para a classificação das amostras.

Por fim, precisamos selecionar as "classes" (tipo tumoral: COAD, READ ou STAD) a serem utilizadas pelo modelo para classificar as amostras

In [ ]:
# open merged file with all the 300 random samples from each tumor type

# read merged .tsv file
df = pd.read_csv("../data/GastroTCGA.tsv", delimiter='\t', index_col=0, header=None)

# extract features - gene expression
X = df.T.iloc[:, 2:].values

# select classes - tumor type
Y = df.T.iloc[:, 1].values

### *Classificação com validação "simples"*

Aqui está o coração do algoritmo (aonde a "mágica" ocorre!). Vamos por partes:

1. Primeiro, separamos nosso conjunto de dados em 2 partes: o conjunto de treino e o de teste. Usualmente 70% dos dados são utilizados para treino e os outros 30% para teste, mas isso pode variar
1. Depois, aplicamos a regressão logística no conjunto de treino para obtermos um modelo ótimo.
1. Então, aplicamos nosso modelo para o conjunto de teste
1. Por fim, julgamos a eficiência de nosso modelo vendo a sua acurácia.

PS: aqui temos alguns conceitos importantes:
- Penalidade: tipo de regularização (L1 - Lasso, L2 - Ridge, Elastic-Net, etc...) para penalizar o modelo e evitar "overfitting"
- "Solver": método para encontrarmos os melhores pesos das características para predizer as classes com o menor erro possível (função de custo)
- "Multi class": temos 3 classes, ou seja, multinomial

#### Perguntas

- Quantas amostras foram corretamente classificadas? Você achou esse modelo eficiente?
- Você percebeu algum viés de classificação? Faz sentido?

In [ ]:
#### Simple Validation ####
# 70% of the data in the training set and 30% in the test set
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=0)

# call logistic regression
logreg = LogisticRegression(penalty='l2', C=0.2, solver='lbfgs', multi_class='multinomial', max_iter = 2000)

# Create an instance of Logistic Regression Classifier and fit the data.
logreg.fit(X_train, Y_train)

# Apply the model on the test set
Y_pred = logreg.predict(X_test)
probs = logreg.predict_proba(X_test)

# Accuracy
print('Misclassified test samples: %d' % (Y_test != Y_pred).sum())
print('Accuracy: %.2f' % metrics.accuracy_score(Y_test, Y_pred))

### Figuras

Para melhor visualizarmos os resultados do nosso modelo, vamos fazer algumas figuras! Nesse exemplo, vamos fazer um heatmap e uma matriz de confusão.

#### Perguntas

- Agora você consegue visualizar o suposto viés de classificação desse modelo? Aonde ele ocorre?
- Você plotaria algum dos resultados mostrados de outra forma?

In [ ]:
# Make some plots!

# Non-normalized confusion matrix
class_names = ['COAD', 'READ', 'STAD']
plot = plot_confusion_matrix(Y_test, Y_pred, classes=class_names,
                      title='Confusion matrix, without normalization')


In [ ]:
# Normalized confusion matrix
plot_normalized = plot_confusion_matrix(Y_test, Y_pred, classes=class_names, normalize=True,
                      title='Normalized confusion matrix')


In [ ]:
# Heatmap
dfProbs = pd.DataFrame(probs)

dfProbs.columns = class_names
dfProbs.index = Y_test

plt.figure(figsize=(10, 200))
sns.clustermap(dfProbs.T, vmax=1, cmap="YlGnBu");

### *Classificação com validação cruzada*

A classificação com validação cruzada é um modelo mais robusto para testar a eficiência de seu classificador, especialmente se não há disponibilidade de um conjunto de dados externo (do ICGC, por exemplo) para você desafiar seu modelo. Vamos por partes:

1. Explicar passo a passo
1. Por fim, novamente julgamos a eficiência de nosso modelo vendo a sua acurácia.

#### Perguntas

- Quantas amostras foram corretamente classificadas?
- Você achou esse modelo mais ou menos eficiente do que o anterior? Por quê?
- Você achou esse modelo mais ou menos confiável? Por quê?
- Você percebeu algum viés de classificação aqui também? Faz sentido?

In [ ]:
##### Cross Validation #####
# evaluate the model using 5-fold cross-validation
scores = cross_val_score(logreg, X, Y, scoring='accuracy', cv=5)
Y_pred = cross_val_predict(logreg, X, Y, cv=5)
probs = cross_val_predict(logreg, X, Y, cv=5, method='predict_proba')

print("Accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

### Figuras

Para melhor visualizarmos os resultados desse nosso novo modelo, vamos fazer algumas figuras! Nesse exemplo, vamos novamente fazer um heatmap e uma matriz de confusão a fim de tornar a comparação entre os resultados dos dois modelos mais intuitiva e fácil de interpretar.

#### Perguntas

- Agora você consegue visualizar o suposto viés de classificação desse modelo? Aonde ele ocorre?
- Você plotaria algum dos resultados mostrados de outra forma?

In [ ]:
# Make some plots!

# Plot non-normalized confusion matrix
plot_cross = plot_confusion_matrix(Y, Y_pred, classes=class_names,
                      title='Confusion matrix, Cross Validation, without normalization')


In [ ]:
# Plot normalized confusion matrix
plot_cross_normalized = plot_confusion_matrix(Y, Y_pred, classes=class_names, normalize=True,
                      title='Normalized confusion matrix, Cross Validation')


In [1]:
# Heatmap
dfProbs = pd.DataFrame(probs)

dfProbs.columns = class_names
dfProbs.index = Y

plt.figure(figsize=(10, 200))
sns.clustermap(dfProbs.T, vmax=1, cmap="YlGnBu");


NameError: ignored

In [ ]:
# Heatmap 2
plt.figure(figsize=(10, 200))
sns.clustermap(dfProbs.T, vmax=1, cmap="YlGnBu", col_cluster=False);
